# CapitalTrust EGP Financial Analyst — System Execution

عرض تنفيذي لمشروع LangGraph: محلّل، مراجع، تصحيح ذاتي، حفظ دائم، وموافقة بشرية.

**Evidence boundaries:** the supplied CBE-style document (2023-11-12) is
unverified training material, the EGX CSV ends on 2024-03-07, and the MCP FX tool
serves an explicitly synthetic quote. Nothing is sent to clients.

The default **demo** backend is deterministic, not an LLM-as-a-Judge.
The **live** backend uses separate structured Gemini analyst/reviewer calls.
Local SQLite audit records are real local evidence, **not Langfuse links**.
For the remaining live acceptance run, configure `.env` as described in `README.md`:
`APP_MODE=live`, `LLM_PROVIDER=gemini`, `AUDIT_MODE=langfuse`,
`EMBEDDING_MODE=gemini` and credentials. A `.env` in the Sprint root is also loaded.

## 1. Setup and explicit operating mode

Run with this project's Python environment. This notebook uses a new runtime
directory on each run, preserving all previous threads. No keys are printed.
The as-of date is the commissioning session date; stale inputs remain labeled.

In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys
from uuid import uuid4
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "src" / "workflow_graph.py").exists():
    raise RuntimeError("Run from the project root or its notebooks/ folder")
sys.path.insert(0, str(ROOT / "src"))

from workflow_graph import Settings
from workflow_graph import AuditLog
from workflow_graph import RAGService
from workflow_graph import load_market_context
from mcp_tools import connect_fx_tool
from state_schema import A2AMessage, HumanDecision
from workflow_graph import initial_state, open_workflow, thread_config

RUNTIME = ROOT / "runtime" / ("nb-" + uuid4().hex[:8])
SETTINGS = Settings.from_env(runtime_dir=RUNTIME, inject_demo_defect=True)
THREAD_ID = "capitaltrust-" + uuid4().hex
CONFIG = thread_config(THREAD_ID)
AS_OF = "2026-11-16"
print("Python:", sys.executable)
print("Backend:", SETTINGS.mode, "| Audit:", SETTINGS.audit_mode,
      "| Embeddings:", SETTINGS.embedding_mode)
print("Provider:", SETTINGS.provider, "| Model:", SETTINGS.model)
print("Runtime:", RUNTIME)
print("Persistent thread ID:", THREAD_ID)
if SETTINGS.mode == "demo":
    print("DEMO: deterministic agent behavior, not live LLM validation.")

## 2. Ingest CBE guidelines, query Chroma and audit every retrieval

TextLoader → overlapping chunks → embeddings → persistent Chroma collection.
Document and embedding configuration hashes identify the collection; repeated
ingestion is idempotent. Local hashed embeddings are a lexical offline baseline.
Gemini semantic embeddings are available in the live configuration.

Every retrieval has callbacks capturing the query and returned context.
In Langfuse mode every action prints its real trace URL. Missing credentials or
failed authentication stops that mode instead of silently falling back.

In [ ]:
audit = AuditLog(RUNTIME / "audit.sqlite")
rag = RAGService(SETTINGS, audit)
print("Chroma collection:", rag.collection_name)
print("Document SHA-256:", rag.document_hash)
print("Indexed chunks:", rag.chunk_count)
sample, sample_audit = await rag.retrieve(
    "Current Corridor Rates Overnight Deposit Rate", THREAD_ID)
for item in sample[:2]:
    print(item["source_id"], "| issued:", item["source_date"])
    print(item["content"][:500], "\n")
if sample_audit["trace_url"]:
    display(Markdown(f"[Open actual Langfuse retrieval trace]({sample_audit['trace_url']})"))
else:
    print("Langfuse acceptance: PENDING credentials; local audit is active.")
assert audit.verify()

## 3. Query historical EGX movements and call the real MCP server

The percentage calculation is **(close / open − 1) × 100**, not an EGX30 index
return. As-of filtering excludes future rows. The MCP connection below uses stdio
discovery and a protocol tool call, even though its financial observation is mock.

In [ ]:
import pandas as pd
market, market_evidence = load_market_context(
    SETTINGS.data_dir / "egx_daily_summaries.csv", AS_OF)
print("Latest CSV session:", market["latest_session"], "| age days:", market["age_days"])
print(market["limitations"])
display(pd.DataFrame(market["rows"])[["Date", "Ticker", "Open", "Close", "Volume", "open_to_close_pct"]])
async with connect_fx_tool() as tool:
    fx_sample = await tool.ainvoke({"base_currency": "USD"})
    print("Discovered MCP tool:", tool.name)
    print(json.dumps(fx_sample, indent=2))
assert fx_sample["is_mock"] and fx_sample["unit"] == "EGP per 1 USD"

## 4. Execute the evaluator–optimizer loop

**Controlled fault injection is enabled for this acceptance demonstration.**
An unsupported citation is deliberately added to the initial draft. The reviewer
must detect it from the evidence; it is not told to approve a particular attempt.
Feedback is sent back in a validated A2A message. The next draft must fix it.

In demo mode these decisions are deterministic checks. In live mode the same
graph uses a structured LLM reviewer plus the deterministic safety checks.

In [ ]:
async with open_workflow(SETTINGS) as (graph, services):
    async for event in graph.astream(
        initial_state("Prepare an evidence-grounded EGP morning brief for director review.", AS_OF),
        CONFIG, stream_mode="updates",
    ):
        for node, update in event.items():
            if node == "analyst":
                print(f"\nANALYST — draft {update['iteration']}")
                print(update["draft_report"])
            elif node == "reviewer":
                print("\nREVIEWER:", "APPROVED" if update["approval_status"] else "REJECTED")
                print(update["reviewer_feedback"])
            elif node == "__interrupt__":
                print("\nHITL: execution paused; waiting for an explicit human command.")
    snapshot = await graph.aget_state(CONFIG)
    paused_state = snapshot.values
    next_nodes = snapshot.next
    assert services["audit"].verify()

assert paused_state["status"] == "pre_publish"
assert next_nodes == ("pre_publish",)
assert paused_state["review_history"][0]["is_approved"] is False
assert paused_state["review_history"][-1]["is_approved"] is True
assert "UNSUPPORTED-DEMO-CITATION" not in paused_state["draft_report"]
assert "human_decision" not in paused_state
print("\nReview results:", [item["is_approved"] for item in paused_state["review_history"]])
print("Approved draft SHA-256:", paused_state["approved_draft_sha256"])
print("No publication has occurred.")

## 5. Validate all handoffs and inspect retrieval evidence

Both the strict Pydantic model and the supplied A2A JSON schema must accept every
handoff. Strings such as `"false"` cannot masquerade as boolean approval values.

In [ ]:
import jsonschema
schema = json.loads((ROOT / "schemas" / "a2a_message_schema.json").read_text(encoding="utf-8"))
for message in paused_state["a2a_messages"]:
    A2AMessage.model_validate(message)
    jsonschema.validate(message, schema)
print("Validated handoffs:", len(paused_state["a2a_messages"]))
for item in paused_state["retrieval_audits"]:
    print(item["retrieval_id"], "| query:", item["query"])
    print("Retrieved:", ", ".join(item["source_ids"]))
    print("Langfuse:", item["trace_url"] or "NOT CONFIGURED — local audit only")
print("Audit chain valid:", audit.verify())
if SETTINGS.audit_mode == "langfuse":
    from workflow_graph import verify_retrieval_traces
    verified_traces = await verify_retrieval_traces(
        rag.langfuse, [sample_audit] + paused_state["retrieval_audits"])
    assert len(verified_traces) == len(paused_state["retrieval_audits"]) + 1
    for verified in verified_traces:
        print("READ-BACK VERIFIED — query and context:", verified["trace_url"])
else:
    verified_traces = []

## 6. Sliding window and summary demonstration

The following synthetic message history exercises prolonged feedback memory.
Only the recent window and compact summary are sent onward. Current reviewer
feedback and current evidence remain separate state fields. Full audit records
are not removed by prompt trimming. The scratchpad contains actions/outcomes,
not hidden model reasoning.

In [ ]:
from workflow_graph import AgentBackend
from workflow_graph import compact_messages
older_messages = [
    {"role": "user", "content": f"Review round {i}: verify evidence and date. " + "context " * 180}
    for i in range(30)
]
kept, summary = await compact_messages(
    older_messages, "Earlier rounds checked citations.", SETTINGS, AgentBackend(SETTINGS))
assert len(kept) <= SETTINGS.window_messages
assert sum(len(m["content"]) for m in kept) <= SETTINGS.context_chars
assert 0 < len(summary) <= SETTINGS.summary_chars
print("Original messages:", len(older_messages), "| retained:", len(kept))
print("Summary characters:", len(summary))
print(summary[:700])
print("Bounded scratchpad entries:", len(paused_state["scratchpad_memory"]))

## 7. Recover the interrupted thread from a fresh Python process

The process below has no access to this kernel's Python objects. It rebuilds the
graph and opens the same SQLite database using the same thread ID. An inspection
must find the exact approved draft still waiting at `pre_publish`.

In [ ]:
SUBPROCESS_ENV = os.environ | {"PYTHONUTF8": "1", "ANONYMIZED_TELEMETRY": "False"}
def process_command(command, thread_id, runtime, extra=None, *, mode=None, audit_mode=None, embedding_mode=None):
    args = [sys.executable, str(ROOT / "src" / "workflow_graph.py"), command,
            "--thread-id", thread_id, "--runtime-dir", str(runtime),
            "--mode", mode or SETTINGS.mode,
            "--audit-mode", audit_mode or SETTINGS.audit_mode,
            "--embedding-mode", embedding_mode or SETTINGS.embedding_mode]
    result = subprocess.run(args + (extra or []), capture_output=True, text=True,
                            encoding="utf-8", env=SUBPROCESS_ENV, timeout=180)
    if result.returncode:
        raise RuntimeError(result.stderr)
    lines = result.stdout.strip().splitlines()
    for line in lines[:-1]:
        print(line)  # Includes each actual retrieval trace URL, when applicable.
    return json.loads(lines[-1])

recovered = process_command("inspect", THREAD_ID, RUNTIME)
assert recovered["process_id"] != os.getpid()
assert recovered["next"] == ["pre_publish"]
assert recovered["state"]["draft_report"] == paused_state["draft_report"]
assert recovered["state"]["approved_draft_sha256"] == paused_state["approved_draft_sha256"]
print("Notebook kernel PID:", os.getpid(), "| fresh process PID:", recovered["process_id"])
print("Recovered thread:", recovered["thread_id"], "| status:", recovered["state"]["status"])
print("Persistent checkpointer:", RUNTIME / "checkpoints.sqlite")

## 8. Complete publish/reject lifecycle in isolated acceptance fixtures

**SIMULATED HUMAN DECISIONS — TEST THREADS ONLY.** These separate demo processes
exercise both resume paths and verify persistence after process termination.
They only write an authorization record. No email, client delivery or external
publication is implemented. The primary thread above stays on hold for a person.

In [ ]:
lifecycle_results = []
fixture_retrievals = []
for action in ("publish", "reject"):
    test_thread = "fixture-" + action + "-" + uuid4().hex[:10]
    test_runtime = RUNTIME / ("p" if action == "publish" else "r")
    opts = dict(mode="demo", audit_mode=SETTINGS.audit_mode, embedding_mode="local")
    started = process_command("start", test_thread, test_runtime,
        ["--inject-demo-defect", "--as-of", AS_OF], **opts)
    assert started["state"]["status"] == "pre_publish"
    finished = process_command("resume", test_thread, test_runtime,
        ["--action", action, "--reviewer", "SIMULATED ACCEPTANCE TEST DIRECTOR",
         "--draft-sha256", started["state"]["approved_draft_sha256"]], **opts)
    assert started["process_id"] != finished["process_id"]
    expected = "published" if action == "publish" else "rejected"
    assert finished["state"]["status"] == expected and finished["next"] == []
    assert not finished["state"]["publication"]["delivered"]
    assert finished["audit_chain_valid"]
    fixture_retrievals.extend(started["state"]["retrieval_audits"])
    lifecycle_results.append({"command": action, "status": expected,
        "start_pid": started["process_id"], "resume_pid": finished["process_id"],
        "thread_id": test_thread, "delivered": False})
    print(json.dumps(lifecycle_results[-1], indent=2))
if SETTINGS.audit_mode == "langfuse":
    verified_fixture_traces = await verify_retrieval_traces(rag.langfuse, fixture_retrievals)
    verified_traces.extend(verified_fixture_traces)
    print("Read-back verified fixture RAG traces:", len(verified_fixture_traces))

## 9. Actual director decision — intentionally left blank

Review the approved draft and its hash before changing the cell below.
Set `HUMAN_ACTION` to `publish` or `reject`, enter your name, and run the cell.
Keeping it blank leaves the real report thread interrupted. If you restart the
kernel, use the printed runtime/thread/hash with `workflow_graph.py inspect/resume`
as described in the README. A wrong hash or malformed decision remains on hold.

In [ ]:
HUMAN_ACTION = ""  # Explicitly enter publish or reject after reviewing the draft.
HUMAN_REVIEWER = ""
if HUMAN_ACTION:
    decision = HumanDecision(action=HUMAN_ACTION, reviewer=HUMAN_REVIEWER,
        draft_sha256=paused_state["approved_draft_sha256"])
    final = process_command("resume", THREAD_ID, RUNTIME,
        ["--action", decision.action, "--reviewer", decision.reviewer,
         "--draft-sha256", decision.draft_sha256])
    print("Human decision recorded:", final["state"]["status"])
else:
    print("WAITING FOR HUMAN — primary thread remains pre_publish.")
    print("thread_id:", THREAD_ID)
    print("runtime_dir:", RUNTIME)
    print("draft_sha256:", paused_state["approved_draft_sha256"])

## 10. Findings and acceptance evidence

- Demonstrated rejection → revision → approval, with validated A2A feedback.
- Demonstrated real MCP discovery/calls serving labeled mock observations.
- Demonstrated persistent Chroma ingestion and per-retrieval callback logs.
- Demonstrated sliding-window trimming and bounded summaries.
- Demonstrated recovery and resume in different OS processes with SQLite.
- Demonstrated publish/reject test commands; the primary thread awaits a person.

**Remaining live evidence:** if this notebook was executed in demo/local mode,
LLM-as-a-Judge behavior and semantic embeddings require live mode. Hosted Langfuse
trace links separately require Langfuse credentials and audit mode. The final
output records exactly which modes were exercised; local logs are not hosted traces.

**Before production:** authenticate/authorize directors at a service boundary;
store secrets securely; authenticate source documents and obtain current feeds;
validate freshness policies; independently evaluate judge accuracy; add durable
delivery/outbox idempotency; externally anchor audit hashes and implement access
controls, retention and backups. A local hash chain alone is not immutable storage.

In [ ]:
acceptance = {
    "backend": SETTINGS.mode,
    "provider": SETTINGS.provider,
    "model": SETTINGS.model,
    "embedding_mode": SETTINGS.embedding_mode,
    "embedding_model": SETTINGS.embedding_model,
    "self_correction": [r["is_approved"] for r in paused_state["review_history"]],
    "all_a2a_validated": True,
    "recovered_in_fresh_process": recovered["process_id"] != os.getpid(),
    "publish_and_reject_fixtures": lifecycle_results,
    "primary_thread": THREAD_ID,
    "primary_status": "human_decision_submitted" if HUMAN_ACTION else "pre_publish",
    "retrievals": len(paused_state["retrieval_audits"]) + 1 + len(fixture_retrievals),
    "langfuse_trace_urls": [r["trace_url"] for r in [sample_audit] + paused_state["retrieval_audits"] + fixture_retrievals if r["trace_url"]],
    "langfuse_read_back_verified": verified_traces,
    "live_llm_validation": "executed" if SETTINGS.mode == "live" else "PENDING credentials/live run",
    "langfuse_validation": "executed" if SETTINGS.audit_mode == "langfuse" else "PENDING credentials/live run",
}
(RUNTIME / "acceptance_evidence.json").write_text(json.dumps(acceptance, indent=2), encoding="utf-8")
print(json.dumps(acceptance, indent=2))